In [2]:
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough
from langchain_community.retrievers import BM25Retriever
from openai import AzureOpenAI
import dotenv
import getpass
import os
import requests
import pathlib   # navigates the file system and opens files
import pprint    # for inspecting Iliad response messages
import json

In [3]:
dotenv.load_dotenv()

ILIAD_API_KEY = os.getenv('ILIAD_API_KEY')
GOOGLE_API_KEY = os.getenv('GOOGLE_API_KEY')
ILIAD_URL_BASE = os.getenv('ILIAD_URL_BASE')

if not os.getenv("ILIAD_API_KEY"):
    os.environ["ILIAD_API_KEY"] = getpass.getpass("Enter your ILIAD API key: ")

def get_access_token():
    response = requests.post(
        url="https://federation.abbvie.com/as/token.oauth2",
        data={
            "grant_type": "client_credentials",
            "client_id": os.environ["CLIENT_ID"],
            "client_secret": os.environ["CLIENT_SECRET"]
        }
    )
    response.raise_for_status()
    data = response.json()
    token = data["access_token"]
    return f"Bearer {token}"

In [1]:
# test MCP server
from langchain_mcp_adapters.client import MultiServerMCPClient
from langchain.agents import create_agent
from langchain_openai import AzureChatOpenAI
from langchain_anthropic import ChatAnthropic
from langchain.messages import HumanMessage, AIMessage, SystemMessage
import langgraph

In [ ]:
# Connect to your R MCP server
# Approach 1: You need to run your server with http for this approach
mcp_client = MultiServerMCPClient(
    {
        "samplesize": {
            "transport": "streamable_http",
            "url": "http://127.0.0.1:9290",
        },
        # "calculator": {
        #     "transport": "streamable_http",
        #     "url": "http://0.0.0.0:8282/mcp",
        # }
    }
)

In [ ]:
# Approach 2: Run from CLI
# mcp_client = MultiServerMCPClient(
#     {
#         "samplesize": {
#             "transport": "stdio",
#             "command": "Rscript",
#             # Absolute path to your R MCP file
#             "args": ["./OASiS/SampleSizeMCP.R"],
#             }
#     }
# )

In [10]:
# Get tools from the MCP server
tools = await mcp_client.get_tools()

# Check your tools and connection
tools

[StructuredTool(name='math_calculator', description='Performs basic arithmetic operations', args_schema={'type': 'object', 'properties': {'operation': {'type': 'string', 'title': 'Operation', 'description': 'Math operation to perform', 'enum': ['add', 'subtract', 'multiply', 'divide']}, 'a': {'type': 'number', 'title': 'First number', 'description': 'First operand'}, 'b': {'type': 'number', 'title': 'Second number', 'description': 'Second operand'}}, 'additionalProperties': False, 'required': ['operation', 'a', 'b']}, response_format='content_and_artifact', coroutine=<function convert_mcp_tool_to_langchain_tool.<locals>.call_tool at 0x7f65f296f600>)]

In [ ]:
# Create a LangChain agent connected to the R MCP Server
llm = AzureChatOpenAI(
    api_key=ILIAD_API_KEY,
    azure_endpoint=ILIAD_URL_BASE,
    openai_api_version="2023-07-01-preview",
    azure_deployment="gpt-4o")

agent = create_agent(
    llm, 
    tools
    # system_prompt = "Pass collected parameters to MCP server. Do not interpret any parameters. According to the parameters and multiplicity procedure provided by user, Calculate the sample size via MCP server. Only display output from Serer. Do not further interpret output."
    )

In [12]:
# test LLM
# message = HumanMessage("What tools in MCP server are you using?")
# response = await agent.ainvoke(message)

# print(response)
response = agent.invoke({"input": "What is 10 multiplied by 5?"})
print(response["output"])

BadRequestError: Error code: 400 - {'error': {'message': "Invalid 'messages': empty array. Expected an array with minimum length 1, but got an empty array instead.", 'type': 'invalid_request_error', 'param': 'messages', 'code': 'empty_array'}}

In [12]:
# Use the agent
SampleSize_response = await agent.ainvoke(
    {
        "messages": 
        [
           {"role": "user", 
            "content": """Please calculate sample size given the following design:
            1. Multiplicity Procedure: Graphical Procedure 
            2. Names of treatment arms: Placebo, ABBV932L, ABBV932H 
            3. Effect sizes of primary efficacy endpoints for treatment arms: -0.33, -0.33
            4. Effect sizes of key secondary endpoints for treatment arms: -0.33, -0.33, -0.28, -0.28
            5. Allocation Ratio: 1:1:1
            6. Drop out rate: 0.2
            7. Between-endpoints Correlation Matrix: [1, 0.6, 0.6,
                                                      0.6, 1, 0.6, 
                                                      0.6, 0.6, 1]
            8. required statistical power: 0.9
            9. Success criteria: 
              1). disjuctive power
              2). weighted power: 0.8 on ABBV932L primary endpoint and 0.2 on ABBV932H primary endpoint
            10. Significance level: 0.05
            11. Initial hypothesis Weights: [0.5, 0.5, 0, 0, 0, 0]
            12. Transition Matrix: [0, 0.5, 0.5, 0, 0, 0,
                                    0.5, 0, 0, 0.5, 0, 0,
                                    0, 0, 0, 0, 1, 0,
                                    0, 0, 0, 0, 0, 1,
                                    0, 0, 0, 0, 0, 0,
                                    0, 0, 0, 0, 0, 0]
            13. Number of simulated trials: 1000
            14. Minimum number of subjects: 100
            15. Maximum number of subjects: 300
            16. Increment of number of subjects: 5
            17. Seed: 12345"""
            }
        ]
    })

# print(SampleSize_response["output"])

McpError: argument of length 0